# 15.10 Logging — Configuration, Handlers and Structured Output

**Prerequisites:** 15.7 Reading Failures, 15.4 Fixtures (`caplog`), 07 Module and Packages  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 Logging goes to **stderr** by default — and why that matters
- The five levels, and the **two gates** every record must pass
- The logger **hierarchy**, propagation, and `getLogger(__name__)`
- 🔴 `basicConfig` silently does nothing the second time
- 🔴 **Double logging** — the most common logging bug there is
- Handlers: files, rotation, and where log records actually go
- `dictConfig` — configuring the whole application in one place
- **Structured logging**: `extra=`, a JSON formatter, and log aggregators
- Libraries and `NullHandler` — never configure logging in a library
- Testing your logging with `caplog` (**15.4**)

---

## Where this fits

**15.7** made the case for logging as a *diagnostic instrument*: `print` answers a question in
the next sixty seconds, logging answers questions you will ask **again**, in production, without
editing code. It covered `logging.exception()`, lazy `%s` formatting and `stacklevel`.

This notebook is the other half — **making logging actually work**. That is almost entirely
about configuration, and configuration is where every logging bug lives.

> **Why this notebook is here and not in a "modern features" folder.** Logging has been in the
> standard library since 2003; it is not a modern feature. It is a **debugging instrument**, and
> it is the one you reach for precisely when the debugger cannot help — intermittent failures,
> production-only failures, and anything concurrent (**15.9**).

Everything runs in a **subprocess**, because logging configuration is global process state:
`basicConfig`, handlers and levels persist for the life of the interpreter. A fresh process per
demo is the only way to show honest behaviour — and it is the same reason logging bugs are so
often "works in the test, breaks in the app".

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py1510_"))


def run_script(name, source, *args, split=False):
    """Run a script in a fresh interpreter.

    🔴 stdout and stderr are returned SEPARATELY when split=True, because
    logging writes to stderr by default and `print` writes to stdout - mixing
    them hides exactly the thing the first demo is about.
    """
    path = WORK / name
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    done = subprocess.run([sys.executable, str(path), *args],
                          capture_output=True, text=True, encoding="utf-8",
                          errors="replace", timeout=120)
    if split:
        return done.stdout.rstrip(), done.stderr.rstrip()
    return (done.stdout + done.stderr).rstrip()


print("scratch:", WORK)

## 🔴 Logging writes to `stderr`, not `stdout`

The very first surprise, and it has real consequences:

- Your log lines **do not appear** in `program.py > output.txt`.
- In a container, `stdout` and `stderr` may go to different places.
- If your program's *output* is data being piped somewhere, logging to `stdout` would
  **corrupt it** — which is exactly why the default is `stderr`.

`basicConfig(stream=sys.stdout)` changes it when you want that.

In [ ]:
out, err = run_script("streams.py", r"""
    import logging
    import sys

    print("this is print() - stdout")
    logging.warning("this is logging - where does it go?")

    logging.basicConfig(stream=sys.stdout, format="%(levelname)s %(message)s", force=True)
    logging.warning("and now, with stream=sys.stdout")
""", split=True)

print("--- STDOUT ---")
print(out)
print("\n--- STDERR ---")
print(err)

Two different destinations, and nothing in the code said so. The first
`logging.warning` went to **stderr** through the *last-resort handler* — the fallback that
exists so a warning is never silently lost when nothing is configured.

## Levels, and the two gates

| Level | Value | Means |
|---|---|---|
| `DEBUG` | 10 | diagnostic detail; off in production |
| `INFO` | 20 | normal progress worth recording |
| `WARNING` | 30 | something is off, we coped — **the default threshold** |
| `ERROR` | 40 | this operation failed |
| `CRITICAL` | 50 | the process cannot continue |

🔴 **A record must pass two independent gates**, and forgetting the second one is why
"I set the level to DEBUG and still see nothing":

```
  log.debug(...)
        │
        ▼
  ┌──────────────┐   too low?  ──> dropped, and no handler ever sees it
  │ LOGGER level │
  └──────┬───────┘
         │ passes
         ▼
  ┌───────────────┐  too low?  ──> this handler skips it (others may not)
  │ HANDLER level │
  └───────┬───────┘
          │ passes
          ▼
      formatted and emitted
```

One logger can feed several handlers at **different** levels — a console showing warnings and a
file recording everything is the standard arrangement.

In [ ]:
print(run_script("gates.py", r"""
    import logging
    import sys

    log = logging.getLogger("svc")
    log.setLevel(logging.DEBUG)               # gate 1: wide open

    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.WARNING)         # gate 2a: narrow
    console.setFormatter(logging.Formatter("console >  %(levelname)-8s %(message)s"))
    log.addHandler(console)

    to_file = logging.StreamHandler(sys.stdout)   # pretending to be a file
    to_file.setLevel(logging.DEBUG)               # gate 2b: everything
    to_file.setFormatter(logging.Formatter("file    >  %(levelname)-8s %(message)s"))
    log.addHandler(to_file)

    log.debug("cache miss for user:7")
    log.warning("retry budget nearly exhausted")
"""))

The `DEBUG` line reached only the file handler; the `WARNING` reached both.
One call, two destinations, two thresholds.

## The hierarchy

Logger names are **dotted paths**, and they form a tree rooted at the root logger. A record
travels *up* the tree, offered to every handler it passes.

```
                       root                      handlers here catch everything
                        │
                       app
                     ╱     ╲
                app.db     app.http
                   │
              app.db.pool
```

🔴 **`logging.getLogger(__name__)` is the idiom**, and the reason is the tree: in
`myapp/db/pool.py`, `__name__` is `"myapp.db.pool"`, so your loggers mirror your package layout
(**07**) for free. That means an operator can turn up the volume on **one subsystem**
(`myapp.db` → `DEBUG`) without drowning in everything else.

Two controls:

| | Effect |
|---|---|
| `logger.setLevel(...)` | the threshold for this logger **and everything under it** that has no level of its own |
| `logger.propagate = False` | records stop here — they do **not** reach ancestors' handlers |

A logger with no explicit level has `NOTSET`, and its **effective** level is inherited from the
nearest ancestor that has one.

In [ ]:
print(run_script("hierarchy.py", r"""
    import logging
    import sys

    root = logging.getLogger()
    root.setLevel(logging.DEBUG)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(name)-13s %(levelname)-8s] %(message)s"))
    root.addHandler(handler)

    app = logging.getLogger("app")
    db = logging.getLogger("app.db")
    pool = logging.getLogger("app.db.pool")

    print("1. a deep logger propagates all the way to root:")
    pool.info("connection acquired")

    print("\n2. raise app.db to WARNING - its CHILDREN are filtered too:")
    db.setLevel(logging.WARNING)
    pool.info("dropped: pool inherits app.db's threshold")
    pool.warning("kept: warning clears the bar")

    print("\n3. propagate=False stops records reaching root's handler:")
    db.setLevel(logging.DEBUG)
    db.propagate = False
    pool.info("this goes nowhere - app.db has no handler of its own")
    print("   (nothing was logged above)")

    print("\n4. own level vs EFFECTIVE level - set ONLY app, and watch it inherit:")
    db.propagate = True
    db.setLevel(logging.NOTSET)
    pool.setLevel(logging.NOTSET)
    app.setLevel(logging.WARNING)
    for logger in (app, db, pool):
        print(f"   {logger.name:12} own={logging.getLevelName(logger.level):8}"
              f" effective={logging.getLevelName(logger.getEffectiveLevel())}")
    print("   ^ two loggers have NO level of their own and still answer WARNING")
"""))

Step 2 is the one that catches people: setting a level on `app.db`
**silenced its child** `app.db.pool`, because the child had no level of its own.

## 🔴 `basicConfig` only works once

`basicConfig` is a convenience that configures the **root** logger — and it does nothing at all
if the root logger already has handlers. No error, no warning. Call it twice with different
settings and the second call is silently ignored.

In [ ]:
print(run_script("basicconfig.py", r"""
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.WARNING,
                        format="FIRST  | %(levelname)-8s %(message)s")
    logging.warning("after the first basicConfig")

    # Someone else, elsewhere, tries to reconfigure:
    logging.basicConfig(stream=sys.stdout, level=logging.DEBUG,
                        format="SECOND | %(levelname)-8s %(message)s")
    logging.debug("did the level become DEBUG?")
    logging.warning("did the format change?")
    print("   ^ still the FIRST format, and the DEBUG line never appeared")

    # force=True (3.8+) removes the existing handlers first.
    logging.basicConfig(stream=sys.stdout, level=logging.DEBUG,
                        format="THIRD  | %(levelname)-8s %(message)s", force=True)
    logging.debug("now it works")
"""))

The second call changed **nothing** — not the level, not the format. In a
real application this shows up as "logging config in `main()` has no effect", because something
imported earlier already called `basicConfig` (or added a handler).

| Fix | When |
|---|---|
| `force=True` | you genuinely want to replace the existing configuration (3.8+) |
| `dictConfig` | 🔴 the real answer for an application — see below |
| don't call it at all | 🔴 **in a library** — see the last section |

## 🔴 Double logging

The most common logging bug in real code: **each record printed two, three, four times.**

The cause is always the same — handlers added more than once to the same logger. A `setup()`
function called twice, a module reimported, a fixture that configures logging per test
(**15.4**), or `basicConfig` plus your own handler both attaching to root.

In [ ]:
print(run_script("double.py", r"""
    import logging
    import sys


    def setup_logging():
        log = logging.getLogger("worker")
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter("%(name)s | %(message)s"))
        log.addHandler(handler)          # 🔴 no guard against being called twice
        log.setLevel(logging.INFO)
        return log


    log = setup_logging()
    log.info("first call to setup_logging")

    log = setup_logging()
    log.info("second call")

    log = setup_logging()
    log.info("third call")

    print(f"\n   handlers attached: {len(logging.getLogger('worker').handlers)}")
    print("   ^ the message count matches the handler count, exactly")
"""))

One, then two, then three copies — the number of duplicates *is* the number
of handlers.

**How to avoid it, in order of preference:**

1. **Configure logging exactly once**, at application startup, with `dictConfig`. Never in a
   module that might be imported twice.
2. If you must be defensive: `if not log.handlers:` before adding, or
   `log.handlers.clear()` first.
3. Check `propagate` — a common variant is a handler on both `app` **and** root, so every
   record is emitted once on the way past each.

🔴 If you ever see doubled lines, count the handlers before doing anything else:
`print(logging.getLogger("name").handlers)`.

## Handlers — where records actually go

| Handler | Sends records to |
|---|---|
| `StreamHandler` | a stream — `stderr` by default |
| `FileHandler` | a single file, appended forever |
| `RotatingFileHandler` | a file, rotated by **size** |
| `TimedRotatingFileHandler` | rotated by **time** — hourly, daily, at midnight |
| `SMTPHandler` | email — 🔴 rate-limit it, or an outage becomes 40,000 emails |
| `SysLogHandler` / `NTEventLogHandler` | the OS log |
| `QueueHandler` + `QueueListener` | a queue — the right answer for threads (**12.2**) |
| `NullHandler` | nowhere — for libraries |

Rotation is the one worth seeing, because "the disk filled up with logs" is a real outage.

In [ ]:
print(run_script("rotate.py", r"""
    import logging
    import logging.handlers
    import tempfile
    from pathlib import Path

    directory = Path(tempfile.mkdtemp())
    log = logging.getLogger("rotating")
    log.setLevel(logging.INFO)

    # Tiny maxBytes so rotation is visible in a few lines.
    handler = logging.handlers.RotatingFileHandler(
        directory / "service.log", maxBytes=200, backupCount=3, encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(message)s"))
    log.addHandler(handler)

    for n in range(40):
        log.info("job build-%03d finished", n)

    handler.close()
    print("files on disk:")
    for path in sorted(directory.iterdir()):
        print(f"   {path.name:18} {path.stat().st_size:4} bytes")
    print("\n   40 messages, capped at 4 files. The oldest were discarded -")
    print("   backupCount is a promise about DISK USAGE, not about history.")
"""))

Four files, ~200 bytes each, and the earliest messages are **gone**. That is
the trade: `backupCount` bounds your disk usage, and anything older is deleted. If you need the
history, ship the logs somewhere (which is what structured logging, below, is for).

## `dictConfig` — configuring an application properly

`basicConfig` is for scripts. For an application you want **one declarative block**, ideally
loaded from a file, that names every formatter, handler and logger. That is
`logging.config.dictConfig`.

```
{
  "version": 1,                          always 1
  "disable_existing_loggers": False,     🔴 True would silence loggers created at import time
  "formatters": { ... },                 how a record becomes text
  "handlers":   { ... },                 where it goes, and its own level
  "loggers":    { ... },                 per-subsystem levels
  "root":       { ... }                  the catch-all
}
```

🔴 **`disable_existing_loggers` defaults to `True`.** Any logger created before the config runs
— which includes every `getLogger(__name__)` at module import — is switched **off**. This is the
second-most-common logging mystery after double logging. Set it to `False`.

In [ ]:
print(run_script("dictconfig.py", r"""
    import logging
    import logging.config

    logging.config.dictConfig({
        "version": 1,
        "disable_existing_loggers": False,          # 🔴 see the note above
        "formatters": {
            "plain": {"format": "%(levelname)-8s %(name)-10s | %(message)s"},
            "detailed": {
                "format": "%(asctime)s %(levelname)-8s %(name)s "
                          "%(filename)s:%(lineno)d | %(message)s",
                "datefmt": "%H:%M:%S",
            },
        },
        "handlers": {
            "console": {
                "class": "logging.StreamHandler",
                "formatter": "plain",
                "level": "INFO",
                "stream": "ext://sys.stdout",       # ext:// resolves a dotted name
            },
        },
        "loggers": {
            # The database subsystem is chatty: only warnings, and do not
            # also send them to root's handler.
            "app.db": {"level": "WARNING", "handlers": ["console"], "propagate": False},
        },
        "root": {"level": "DEBUG", "handlers": ["console"]},
    })

    logging.getLogger("app").info("app info - shown")
    logging.getLogger("app.db").info("db info - filtered by the app.db logger")
    logging.getLogger("app.db").warning("db warning - shown ONCE (propagate=False)")
    logging.getLogger("app.http").debug("http debug - root is DEBUG, so shown")
"""))

One block, three different behaviours per subsystem — and `propagate: False`
on `app.db` is what stops its warning appearing twice (once from its own handler, once from
root's).

In a real project this dictionary lives in a **YAML or TOML file** loaded at startup, so
operators can raise `app.db` to `DEBUG` at 3 a.m. without a deployment. `tomllib` is in the
stdlib since 3.11 (**15.6**).

## Structured logging

A human reads `"job build-42 failed on attempt 2"`. A **log aggregator** cannot — it would have
to parse English. Structured logging emits **fields**, so you can query
`job_id="build-42" AND level="ERROR"`.

The mechanism is `extra=`, which attaches arbitrary attributes to the `LogRecord`, plus a
formatter that renders them.

In [ ]:
print(run_script("structured.py", r"""
    import json
    import logging
    import sys


    # Render each record as one JSON object - one line, machine-readable.
    class JsonFormatter(logging.Formatter):

        CUSTOM_FIELDS = ("job_id", "attempt", "region")

        def format(self, record):
            payload = {
                "level": record.levelname,
                "logger": record.name,
                "message": record.getMessage(),      # applies the %s arguments
                "line": record.lineno,
            }
            for field in self.CUSTOM_FIELDS:
                if hasattr(record, field):
                    payload[field] = getattr(record, field)
            if record.exc_info:
                payload["error"] = self.formatException(record.exc_info).splitlines()[-1]
            return json.dumps(payload)


    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(JsonFormatter())
    log = logging.getLogger("worker")
    log.addHandler(handler)
    log.setLevel(logging.INFO)

    log.info("job started", extra={"job_id": "build-42", "attempt": 1, "region": "eu"})
    try:
        int("thirty")
    except ValueError:
        log.exception("job failed", extra={"job_id": "build-42", "attempt": 2})
"""))

Two queryable records. Note `record.getMessage()` — that applies the lazy
`%s` arguments from **15.7**, which is why the formatter must call it rather than reading
`record.msg`.

### 🔴 `extra` cannot overwrite a reserved field

`extra` keys become attributes on the `LogRecord`, so they collide with the built-in ones —
`message`, `name`, `levelname`, `lineno`, `args`, `exc_info` and the rest. Python refuses,
loudly, at the call site.

In [ ]:
print(run_script("collide.py", r"""
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, format="%(message)s")
    log = logging.getLogger("collide")

    for bad_key in ("message", "name", "args"):
        try:
            log.warning("hello", extra={bad_key: "hijacked"})
        except KeyError as exc:
            print(f"   extra={{{bad_key!r}: ...}}  ->  KeyError: {exc}")

    log.warning("a safe key works fine", extra={"job_id": "build-1"})
"""))

> **Third-party options.** `structlog` and `python-json-logger` do the above with
> more features (context binding, processors). The stdlib version here is enough to understand
> what they are doing, and enough for most services.

## Libraries: `NullHandler`, and never configure

🔴 **A library must not configure logging.** Choosing handlers, levels or formats is the
*application's* decision — a library that calls `basicConfig` hijacks it for everybody.

But a library that logs with no handler anywhere hits the **last-resort handler**: the message
goes to `stderr`, unformatted, which surprises users. The convention:

```python
# mylib/__init__.py
import logging
logging.getLogger(__name__).addHandler(logging.NullHandler())
```

That gives the library a handler that discards, so nothing is printed unless the application
opts in — and the moment it does, every `mylib.*` record flows normally.

In [ ]:
out, err = run_script("library.py", r"""
    import logging

    print("A. a library that logs, with nothing configured anywhere:")
    logging.getLogger("noisylib").warning("I decided you needed to see this")

    print("B. the same, but the library added a NullHandler:")
    quiet = logging.getLogger("quietlib")
    quiet.addHandler(logging.NullHandler())
    quiet.warning("you will never see this unless you ask")
    print("   (silent - as it should be)")
""", split=True)

print("--- STDOUT ---")
print(out)
print("\n--- STDERR ---")
print(err or "(empty)")
print("\n^ 'noisylib' reached stderr via the last-resort handler.")
print("  'quietlib' produced nothing at all.")

## Testing that you log the right things

Logging is behaviour, so it can be tested — and sometimes should be. A warning when a retry
budget is exhausted is a *feature*; an operator depends on it.

`caplog` from **15.4** captures records so you can assert on them.

In [ ]:
(WORK / "pt").mkdir(exist_ok=True)
(WORK / "pt" / "test_logging.py").write_text(textwrap.dedent(r"""
    import logging

    log = logging.getLogger("worker")


    def requeue(job_id, attempts, max_attempts=3):
        if attempts >= max_attempts:
            log.warning("job %s exhausted its retry budget", job_id)
            return False
        log.info("requeueing %s (attempt %d)", job_id, attempts + 1)
        return True


    def test_exhausted_budget_warns_once(caplog):
        with caplog.at_level(logging.WARNING, logger="worker"):
            assert requeue("build-1", attempts=3) is False

        assert len(caplog.records) == 1
        record = caplog.records[0]
        assert record.levelname == "WARNING"
        assert record.getMessage() == "job build-1 exhausted its retry budget"


    def test_happy_path_does_not_warn(caplog):
        with caplog.at_level(logging.WARNING, logger="worker"):
            assert requeue("build-1", attempts=0) is True

        assert caplog.records == []


    def test_the_message_is_lazily_formatted(caplog):
        # 🔴 record.msg keeps the TEMPLATE; getMessage() applies the args.
        with caplog.at_level(logging.INFO, logger="worker"):
            requeue("build-7", attempts=1)

        record = caplog.records[0]
        assert record.msg == "requeueing %s (attempt %d)"
        assert record.args == ("build-7", 2)
        assert record.getMessage() == "requeueing build-7 (attempt 2)"
""").lstrip(), encoding="utf-8")

done = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header"],
    cwd=WORK / "pt", capture_output=True, text=True,
    encoding="utf-8", errors="replace", timeout=180)
print((done.stdout + done.stderr).rstrip())

The third test is the interesting one: `record.msg` is still the **template**
and `record.args` holds the values. That is the lazy formatting from **15.7**, visible from the
outside — and it is why asserting on `record.getMessage()` is right while asserting on
`record.msg` tests something else entirely.

> 🔴 Do not test every log line. Test the ones that are **contracts**: an operator alert, an
> audit record, a security event. Asserting on `DEBUG` chatter is the logging equivalent of
> asserting on calls (**15.5**) — it breaks on every refactor and protects nothing.

## Performance and concurrency

| Concern | What to do |
|---|---|
| Cost of disabled levels | `log.debug("...%s", x)`, never an f-string (**15.7**) |
| An expensive argument | guard with `if log.isEnabledFor(logging.DEBUG):` |
| Threads (**12.2**) | logging is thread-safe; handlers hold a lock |
| A **slow** handler blocking workers | `QueueHandler` + `QueueListener` — workers enqueue, one thread writes |
| Multiple **processes** (**12.3**) | 🔴 several processes appending to one file **will interleave and corrupt**; use a queue, a socket handler, or one file per process |
| Async (**12.5**) | the same problem — a blocking handler stalls the event loop |

`QueueHandler` is the standard answer for anything with workers: the logging call becomes a
queue append, and a single listener thread does the slow write.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Calling `basicConfig` twice** and expecting the second to win. It is a silent no-op once the root logger has handlers. Use `force=True`, or `dictConfig`.
2. 🔴 **Adding handlers more than once**, giving duplicated output. Count the handlers before debugging anything else.
3. 🔴 **`disable_existing_loggers` left at its default of `True`** in `dictConfig`, which silences every logger created at import time.
4. **Configuring logging inside a library.** That is the application's decision; add a `NullHandler` and nothing else.
5. **Expecting logs on stdout.** The default is `stderr`, so they vanish from `prog > out.txt`.
6. **Setting the logger level and forgetting the handler level** — or the reverse. A record must clear both.
7. **Using an f-string in a log call**, which formats even when the level is off (**15.7**).
8. **`log.error(str(exc))`** instead of `log.exception(...)`, throwing away the traceback.
9. **Several processes writing to one log file.** Lines interleave and records are corrupted; use a queue or one file per process.
10. **Logging secrets.** Passwords, tokens and card numbers in a log are a breach, and logs are copied everywhere. Redact at the formatter if you cannot trust call sites.
11. **`logging.warn()`** — removed in Python 3.13. It is `logging.warning()`.

## Best Practices

- Use `logging.getLogger(__name__)` in every module, so logger names mirror your packages.
- Configure logging **once**, at application startup, with `dictConfig` — ideally from a file.
- Always set `disable_existing_loggers: False` unless you specifically want the opposite.
- In libraries, add a `NullHandler` and configure nothing.
- Log with `%s` placeholders and pass the values as arguments.
- Use `log.exception(...)` inside `except` blocks; `exc_info=True` elsewhere.
- Attach identifiers with `extra=` and render JSON when logs are aggregated.
- Bound your disk with rotation, and ship anything you need to keep.
- Use `QueueHandler` when workers must not block on a slow handler.
- Test the log lines that are contracts, with `caplog` — and only those.

## Practice Exercises

Try these before moving on.

1. 🔴 Reproduce double logging: call a `setup_logging()` twice and confirm the duplicate count matches the handler count. Then fix it two different ways.
2. Configure a logger with a console handler at `WARNING` and a file handler at `DEBUG`. Prove a `DEBUG` record reaches only one of them.
3. Write a `dictConfig` with `disable_existing_loggers: True`, create a logger *before* applying it, and watch that logger go silent. Explain why to someone else.
4. Take the `Job` class from **15.2** and add logging at three levels. Then write `caplog` tests for only the lines you would call contracts.
5. Write a `RedactingFormatter` that replaces anything looking like a token (**09**, regex) with `***` before the record is emitted.
6. Set up a `RotatingFileHandler` with `maxBytes=1024, backupCount=2`, write 500 lines, and work out from the files how many messages survived.
7. 🔴 Start four threads (**12.2**) all logging to one `StreamHandler`. Then do the same with four *processes* (**12.3**) appending to one file. Which output is corrupted, and why does the thread version survive?
8. Extend the `JsonFormatter` to include the timestamp in ISO-8601 UTC and the thread name. Which `LogRecord` attributes did you need? (`logging` docs list them all.)
9. **Interview question:** an application's logs each appear three times. What are the three most likely causes, and how would you distinguish them in under a minute?

---

## Version notes

| Version | Change |
|---|---|
| **3.13** | 🔴 `logging.warn()` **removed** — deprecated since 3.3; use `logging.warning()` |
| **3.12** | `logging.getLevelNamesMapping()` added; `dictConfig` accepts callable filters |
| **3.11** | `logging.getLevelNamesMapping` groundwork; `tomllib` in the stdlib, so config can live in `pyproject.toml` |
| **3.8** | 🔴 `basicConfig(force=True)` — the escape hatch used above; `stacklevel=` on logging calls (**15.7**) |
| **3.2** | `dictConfig` introduced, superseding the older `fileConfig` (which you will still meet in old projects) |

## 15 Testing and Debugging — the folder

| Notebook | Covers |
|---|---|
| **15.1–15.6** | testing: `assert`, `unittest`, `pytest`, fixtures, doubles, coverage and properties |
| **15.7** | tracebacks, chained exceptions, `excepthook`, `faulthandler` — and *why* to log |
| **15.8** | `pdb`, `breakpoint()`, post-mortem, `pytest --pdb` |
| **15.9** | debugging method, bisection, delta debugging, heisenbugs |
| **15.10** | this notebook — *how* to log: configuration, handlers, structure |

## Related

- **15.7 Reading Failures** — `logging.exception()`, lazy formatting, print vs logging vs debugger
- **15.4 Fixtures** — `caplog`, used here to test logging
- **15.9** — why logging is the tool for intermittent and production-only bugs
- **07 Module and Packages** — `__name__`, which gives the logger hierarchy its shape
- **12.2 / 12.3 Concurrency** — threads, processes, and why one log file is not enough
- **09 Regular Expression** — for a redacting formatter